# 07 — USA Central Valley, NAIP 1 m with Delineate-Anything

Delineates large commercial agricultural fields in California's Central Valley using **NAIP** imagery at 1 m resolution and the **Delineate-Anything** (YOLO) engine.

**Estimated runtime:** ~20–40 minutes (1 year, 1 m resolution = large raster, GPU).

**Prerequisites:**
```bash
pip install agribound[gee,delineate-anything]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
from pathlib import Path

import agribound

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

OUTPUT_DIR = Path("../../outputs/usa_central_valley")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = "naip"
ENGINE = "delineate-anything"
YEAR = 2022

## Create Study Area

AOI near Fresno, CA — large commercial agriculture.

In [ ]:
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [-119.85, 36.70],
                        [-119.75, 36.70],
                        [-119.75, 36.78],
                        [-119.85, 36.78],
                        [-119.85, 36.70],
                    ]
                ],
            },
            "properties": {"name": "Central Valley AOI"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "central_valley_aoi.geojson")
with open(study_area, "w") as f:
    json.dump(aoi, f)

print(f"Study area: {study_area}")

## Run Delineation

In [ ]:
output_path = OUTPUT_DIR / f"fields_naip_{YEAR}.gpkg"

gdf = agribound.delineate(
    study_area=study_area,
    source=SOURCE,
    year=YEAR,
    engine=ENGINE,
    output_path=str(output_path),
    gee_project=GEE_PROJECT,
    min_area=10000,
    simplify=3.0,
    n_workers=8,
)

print(f"Delineated {len(gdf)} fields")
if "metrics:area" in gdf.columns:
    area_ha = gdf["metrics:area"].sum() / 10000
    print(f"Total agricultural area: {area_ha:,.1f} ha")
    print(f"Average field size: {gdf['metrics:area'].mean() / 10000:,.1f} ha")

## Visualization

In [ ]:
m = agribound.show_boundaries(
    gdf,
    basemap="Google.Satellite",
    output_html=str(OUTPUT_DIR / "map_central_valley.html"),
)
m